Import + helper (Cross Attention Fusion Version)

In [13]:
import json
import random
import sys
from copy import deepcopy
from pathlib import Path
from typing import Dict, Optional

import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

def find_project_root(start_path: Path, required_dir: str = "src") -> Path:
    current = start_path.resolve()
    while True:
        if (current / required_dir).exists():
            return current
        if current.parent == current:
            raise FileNotFoundError(f"Cannot find project root containing '{required_dir}'")
        current = current.parent

PROJECT_ROOT = find_project_root(Path.cwd(), required_dir="src")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.datasets.multimodal_raw_dataset import (
    MultiModalRawDataset,
    multimodal_raw_collate_fn,
)
from src.models.multimodal.multimodal_text_guided_pvd_infonce_supcon import (
    MultimodalTextGuidedPVDInfoNCESupCon,
)
from src.models.multimodal.multimodal_cross_attn_classifier import (
    MultimodalCrossAttnClassifier
)
from src.models.losses.wrapper_loss import InfoNCESupConLoss

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Imports OK")

def set_random_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

PROJECT_ROOT: /content/drive/MyDrive/Documents/GR1/MulCo-PlantNet
Imports OK


Config

In [14]:
def get_training_config():
    return {
        "data": {
            "train_image_root": "data/AIDG/dataset_PlantDoc/images/train",
            "train_caption_root": "data/AIDG/captions_LLaVA/train",
            "val_image_root": "data/AIDG/dataset_PlantDoc/images/test",
            "val_caption_root": "data/AIDG/captions_LLaVA/test",
            "img_size": 224,
            "use_depth_suppressed": False,
            "strict_caption_match": True,
        },
        "model": {
            "num_classes": 28,
            "clip_model_name": "ViT-L-14",
            "clip_pretrained": "openai",
            "text_input_dim": 768,
            "image_input_dim": 1024,
            "proj_dim": 512,
            "proj_hidden_dim": 768,
            "pvd_hidden_dim": 768,
            "cls_hidden_dim": 1024,
            "dropout": 0.3,
            "normalize_projection": True,
            "freeze_convnext_backbone": True,
        },
        "loss": {
            "ce_weight": 1.0,
            "itc_weight": 0.2,
            "supcon_weight": 0.2,
            "itc_temperature": 0.07,
            "supcon_temperature": 0.07,
        },
        "optimizer": {
            "lr": 1e-4,
            "weight_decay": 1e-4,
        },
        "scheduler": {
            "mode": "max",
            "factor": 0.5,
            "patience": 3,
            "min_lr": 1e-6,
        },
        "training": {
            "seed": 42,
            "device": "auto",
            "batch_size": 2,
            "num_workers": 0,
            "epochs": 30,
            "save_dir": "archive/text_guided_cross_attn_fusion",
            "monitor_metric": "accuracy",
            "monitor_mode": "max",
            "early_stopping_patience": 7,
            "early_stopping_min_delta": 0.0,
            "save_best": True,
            "save_last": True,
        },
    }

config = get_training_config()
config

{'data': {'train_image_root': 'data/AIDG/dataset_PlantDoc/images/train',
  'train_caption_root': 'data/AIDG/captions_LLaVA/train',
  'val_image_root': 'data/AIDG/dataset_PlantDoc/images/test',
  'val_caption_root': 'data/AIDG/captions_LLaVA/test',
  'img_size': 224,
  'use_depth_suppressed': False,
  'strict_caption_match': True},
 'model': {'num_classes': 28,
  'clip_model_name': 'ViT-L-14',
  'clip_pretrained': 'openai',
  'text_input_dim': 768,
  'image_input_dim': 1024,
  'proj_dim': 512,
  'proj_hidden_dim': 768,
  'pvd_hidden_dim': 768,
  'cls_hidden_dim': 1024,
  'dropout': 0.3,
  'normalize_projection': True,
  'freeze_convnext_backbone': True},
 'loss': {'ce_weight': 1.0,
  'itc_weight': 0.2,
  'supcon_weight': 0.2,
  'itc_temperature': 0.07,
  'supcon_temperature': 0.07},
 'optimizer': {'lr': 0.0001, 'weight_decay': 0.0001},
 'scheduler': {'mode': 'max', 'factor': 0.5, 'patience': 3, 'min_lr': 1e-06},
 'training': {'seed': 42,
  'device': 'auto',
  'batch_size': 2,
  'num_wor

In [15]:
class EarlyStopping:
    def __init__(self, patience=10, mode="max", min_delta=0.0):
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(self, current_score):
        if self.best_score is None:
            self.best_score = current_score
            return True

        improved = current_score > self.best_score + self.min_delta if self.mode == "max" else current_score < self.best_score - self.min_delta
        if improved:
            self.best_score = current_score
            self.counter = 0
            return True

        self.counter += 1
        if self.counter >= self.patience:
            self.should_stop = True
        return False

def resolve_device(device_name: str):
    if device_name == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    return device_name

def build_transforms(img_size: int):
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_transform, val_transform

def save_checkpoint(path, model, optimizer, scheduler, epoch, best_metric, history, config):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "best_metric": best_metric,
        "history": history,
        "config": config,
    }
    torch.save(checkpoint, path)

def load_checkpoint(path, model, optimizer=None, scheduler=None, map_location="cpu"):
    checkpoint = torch.load(path, map_location=map_location)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None and checkpoint.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    return checkpoint


In [16]:
def create_datasets_and_loaders(config: Dict):
    train_transform, val_transform = build_transforms(config["data"]["img_size"])

    def resolve_existing_root(preferred_root: str, candidates=("validation", "val", "test")) -> Path:
        preferred_path = PROJECT_ROOT / preferred_root
        if preferred_path.exists():
            return preferred_path
        for candidate in candidates:
            candidate_root = preferred_root.replace("/validation", f"/{candidate}")
            candidate_root = candidate_root.replace("\\validation", f"\\{candidate}")
            candidate_path = PROJECT_ROOT / candidate_root
            if candidate_path.exists():
                print(f"[WARN] Missing root: {preferred_path}")
                print(f"[INFO] Fallback root: {candidate_path}")
                return candidate_path
        raise FileNotFoundError(f"Root not found (checked validation/val/test): {preferred_path}")

    train_dataset = MultiModalRawDataset(
        image_root=PROJECT_ROOT / config["data"]["train_image_root"],
        caption_root=PROJECT_ROOT / config["data"]["train_caption_root"],
        transform=train_transform,
        use_depth_suppressed=config["data"]["use_depth_suppressed"],
        strict_caption_match=config["data"]["strict_caption_match"],
    )
    val_dataset, val_loader = None, None
    if config["data"].get("val_image_root") and config["data"].get("val_caption_root"):
        val_image_root = resolve_existing_root(config["data"]["val_image_root"])
        val_caption_root = resolve_existing_root(config["data"]["val_caption_root"])
        val_dataset = MultiModalRawDataset(
            image_root=val_image_root,
            caption_root=val_caption_root,
            transform=val_transform,
            use_depth_suppressed=config["data"]["use_depth_suppressed"],
            strict_caption_match=config["data"]["strict_caption_match"],
        )
        val_loader = DataLoader(val_dataset, batch_size=config["training"]["batch_size"], shuffle=False, num_workers=config["training"]["num_workers"], pin_memory=torch.cuda.is_available(), collate_fn=multimodal_raw_collate_fn)
    train_loader = DataLoader(train_dataset, batch_size=config["training"]["batch_size"], shuffle=True, num_workers=config["training"]["num_workers"], pin_memory=torch.cuda.is_available(), collate_fn=multimodal_raw_collate_fn)
    return train_dataset, val_dataset, train_loader, val_loader


In [17]:
def create_training_components(config: Dict):
    device = resolve_device(config["training"]["device"])

    # 1. Khởi tạo mô hình chuẩn như bình thường
    model = MultimodalTextGuidedPVDInfoNCESupCon(
        num_classes=config["model"]["num_classes"],
        clip_model_name=config["model"]["clip_model_name"],
        clip_pretrained=config["model"]["clip_pretrained"],
        text_input_dim=config["model"]["text_input_dim"],
        image_input_dim=config["model"]["image_input_dim"],
        proj_dim=config["model"]["proj_dim"],
        proj_hidden_dim=config["model"]["proj_hidden_dim"],
        pvd_hidden_dim=config["model"]["pvd_hidden_dim"],
        cls_hidden_dim=config["model"]["cls_hidden_dim"],
        dropout=config["model"]["dropout"],
        normalize_projection=config["model"]["normalize_projection"],
        device=device,
    ).to(device)

    # 2. TIÊM MODULE CROSS-ATTENTION VÀO BACKBONE CHÍNH
    model.fusion_classifier = MultimodalCrossAttnClassifier(
        image_input_dim=config["model"]["image_input_dim"],
        text_input_dim=config["model"]["text_input_dim"],
        proj_dim=config["model"]["proj_dim"],
        proj_hidden_dim=config["model"]["proj_hidden_dim"],
        pvd_hidden_dim=config["model"]["pvd_hidden_dim"],
        cls_hidden_dim=config["model"]["cls_hidden_dim"],
        num_classes=config["model"]["num_classes"],
        dropout=config["model"]["dropout"],
        normalize_projection=config["model"]["normalize_projection"]
    ).to(device)
    print("[INFO] Injected Token-level Cross-Attention Fusion successfully!")

    if config["model"].get("freeze_convnext_backbone", False):
        for param in model.image_encoder.model.parameters():
            param.requires_grad = False
        for module in [model.image_encoder.tgcbam1, model.image_encoder.tgcbam2, model.image_encoder.tgcbam3, model.image_encoder.tgcbam4]:
            for param in module.parameters():
                param.requires_grad = True
        for param in model.image_encoder.norm4.parameters():
            param.requires_grad = True

    criterion = InfoNCESupConLoss(
        ce_weight=config["loss"]["ce_weight"],
        itc_weight=config["loss"]["itc_weight"],
        supcon_weight=config["loss"]["supcon_weight"],
        itc_temperature=config["loss"]["itc_temperature"],
        supcon_temperature=config["loss"]["supcon_temperature"],
    )

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=config["optimizer"]["lr"], weight_decay=config["optimizer"]["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode=config["scheduler"]["mode"], factor=config["scheduler"]["factor"], patience=config["scheduler"]["patience"], min_lr=config["scheduler"]["min_lr"])

    return model, criterion, optimizer, scheduler, device


In [18]:
model, criterion, optimizer, scheduler, device = create_training_components(config)
trainable_params_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable Params: {trainable_params_count:,}")

[INFO] Injected Token-level Cross-Attention Fusion successfully!
Trainable Params: 7,621,020


In [19]:
@torch.no_grad()
def validate_epoch(model, dataloader, criterion, device="cpu"):
    model.eval()
    total_samples, total_correct, running_loss = 0, 0, 0.0
    for batch in tqdm(dataloader, desc="Validating", leave=False):
        images, texts, labels = batch["image"].to(device), batch["text"], batch["label"].to(device)
        outputs = model(images, texts)
        loss_dict = criterion(outputs, labels)
        preds = torch.argmax(outputs["logits"], dim=1)
        batch_size = labels.size(0)
        total_samples += batch_size
        total_correct += (preds == labels).sum().item()
        running_loss += loss_dict["loss"].item() * batch_size
    return {"loss": running_loss / total_samples, "accuracy": total_correct / total_samples} if total_samples > 0 else {"loss": 0, "accuracy": 0}

def train_epoch(model, dataloader, criterion, optimizer, device="cpu"):
    model.train()
    total_samples, total_correct, running_loss = 0, 0, 0.0
    for batch in tqdm(dataloader, desc="Training", leave=False):
        images, texts, labels = batch["image"].to(device), batch["text"], batch["label"].to(device)
        outputs = model(images, texts)
        loss_dict = criterion(outputs, labels)
        loss = loss_dict["loss"]
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        preds = torch.argmax(outputs["logits"], dim=1)
        batch_size = labels.size(0)
        total_samples += batch_size
        total_correct += (preds == labels).sum().item()
        running_loss += loss.item() * batch_size
    return {"loss": running_loss / total_samples, "accuracy": total_correct / total_samples} if total_samples > 0 else {"loss": 0, "accuracy": 0}


In [ ]:
def train_text_guided_contrastive(config: Dict):
    set_random_seed(config["training"]["seed"])
    device = resolve_device(config["training"]["device"])
    save_dir = PROJECT_ROOT / config["training"]["save_dir"]
    save_dir.mkdir(parents=True, exist_ok=True)

    train_dataset, val_dataset, train_loader, val_loader = create_datasets_and_loaders(config)
    model, criterion, optimizer, scheduler, device = create_training_components(config)

    early_stopping = EarlyStopping(patience=config["training"]["early_stopping_patience"], mode=config["training"]["monitor_mode"])
    history = []
    best_metric = None

    for epoch in range(1, config["training"]["epochs"] + 1):
        print(f"\nEpoch [{epoch}/{config['training']['epochs']}]")
        train_metrics = train_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = validate_epoch(model, val_loader, criterion, device) if val_loader else None
        
        current_metric = val_metrics[config["training"]["monitor_metric"]] if val_loader else train_metrics[config["training"]["monitor_metric"]]
        scheduler.step(current_metric)
        is_best = early_stopping.step(current_metric)
        best_metric = current_metric if best_metric is None else max(best_metric, current_metric)

        history.append({"epoch": epoch, "train": train_metrics, "validation": val_metrics})
        print(f"Train | loss={train_metrics['loss']:.4f} acc={train_metrics['accuracy']:.4f}")
        if val_metrics:
            print(f"Val   | loss={val_metrics['loss']:.4f} acc={val_metrics['accuracy']:.4f}")

        save_checkpoint(save_dir / "last_model.pt", model, optimizer, scheduler, epoch, best_metric, history, config)
        if is_best:
            save_checkpoint(save_dir / "best_model.pt", model, optimizer, scheduler, epoch, best_metric, history, config)
            print(f"[INFO] Saved best model at epoch {epoch}")
        if early_stopping.should_stop:
            print("[INFO] Early stopping triggered!")
            break

    with open(save_dir / "history.json", "w") as f:
        json.dump(history, f, indent=2)
    return history

history = train_text_guided_contrastive(config)


[MultiModalRawDataset] Total selected images: 2336
[MultiModalRawDataset] Valid samples: 2336
[MultiModalRawDataset] Skipped missing caption: 0
[MultiModalRawDataset] Skipped invalid caption: 0
[MultiModalRawDataset] Num classes: 28
[MultiModalRawDataset] class_to_idx: {'Apple_Scab_Leaf': 0, 'Apple_leaf': 1, 'Apple_rust_leaf': 2, 'Bell_pepper_leaf': 3, 'Bell_pepper_leaf_spot': 4, 'Blueberry_leaf': 5, 'Cherry_leaf': 6, 'Corn_Gray_leaf_spot': 7, 'Corn_leaf_blight': 8, 'Corn_rust_leaf': 9, 'Peach_leaf': 10, 'Potato_leaf_early_blight': 11, 'Potato_leaf_late_blight': 12, 'Raspberry_leaf': 13, 'Soyabean_leaf': 14, 'Squash_Powdery_mildew_leaf': 15, 'Strawberry_leaf': 16, 'Tomato_Early_blight_leaf': 17, 'Tomato_Septoria_leaf_spot': 18, 'Tomato_leaf': 19, 'Tomato_leaf_bacterial_spot': 20, 'Tomato_leaf_late_blight': 21, 'Tomato_leaf_mosaic_virus': 22, 'Tomato_leaf_yellow_virus': 23, 'Tomato_mold_leaf': 24, 'Tomato_two_spotted_spider_mites_leaf': 25, 'grape_leaf': 26, 'grape_leaf_black_rot': 27}


Training:   0%|          | 0/1168 [00:00<?, ?it/s]

Validating:   0%|          | 0/118 [00:00<?, ?it/s]

Train | loss=2.3617 acc=0.3378
Val   | loss=1.4022 acc=0.5212
[INFO] Saved best model at epoch 1

Epoch [2/30]


Training:   0%|          | 0/1168 [00:00<?, ?it/s]

...

Train | loss=1.1310 acc=0.6374
Val   | loss=1.0452 acc=0.6822
[INFO] Saved best model at epoch 7

Epoch [8/30]


Training:   0%|          | 0/1168 [00:00<?, ?it/s]

Validating:   0%|          | 0/118 [00:00<?, ?it/s]

Train | loss=1.1069 acc=0.6404
Val   | loss=1.0031 acc=0.6949
[INFO] Saved best model at epoch 8

Epoch [9/30]


Training:   0%|          | 0/1168 [00:00<?, ?it/s]